# **Inteligência Artificial Aplicada - 2026/1**

## **Trabalho 2: Regressor Logístico para Classificação de Câncer no Ovário**

**Professor:** Cícero Ferreira Fernandes Costa Filho  
**Aluno(a):** Maria Giovanna Gonçalves Sales - 22251138

## Importação das bibliotecas

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn import preprocessing
from sklearn.feature_selection import SelectKBest, f_classif
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# altere o caminho dos arquivos conforme necessário para rodar a célula
inputs  = pd.read_excel('ovarianInputs.xlsx', header=None)
targets = pd.read_excel('ovarianTargets.xls', engine='xlrd', header=None)

# salvando dados como csv
inputs.to_csv('ovarianInputs.csv', index=False, header=False)
targets.to_csv('ovarianTargets.csv', index=False, header=False)

print(f"ovarianInputs.csv - {inputs.shape[0]} linhas x {inputs.shape[1]} colunas")
print(f"ovarianTargets.csv - {targets.shape[0]} linhas x {targets.shape[1]} colunas")

FileNotFoundError: [Errno 2] No such file or directory: 'ovarianInputs.xlsx'

## Carregamento dos dados

O arquivo `ovarianInputs.csv` contém uma matriz **216×100** (intensidades de íons), e `ovarianTargets.csv` contém uma matriz **216×2**, onde `[1, 0]` indica câncer e `[0, 1]` indica paciente normal.

In [ ]:
inputs_csv = pd.read_csv('ovarianInputs.csv', header=None)
targets_csv = pd.read_csv('ovarianTargets.csv', header=None)

X = inputs_csv.values # matriz de features (216 x 100)
y = targets_csv.iloc[:, 0].values # coluna 0: 1 = câncer, 0 = normal

print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")
print(f"Pacientes com câncer: {y.sum()}")
print(f"Pacientes normais: {(y==0).sum()}")

## Pré-processamento: Normalização Min-Max

Vamos normalizar as features para o intervalo [0, 1] para melhorar a convergência da regressão logística.

In [ ]:
scaler = preprocessing.MinMaxScaler()
X_scaled = scaler.fit_transform(X)

## Divisão treino/teste

Os dados serão divididos em 80% para treino e 20% para teste.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Treino: {X_train.shape[0]} amostras")
print(f"Teste: {X_test.shape[0]} amostras")

## Criação do modelo

A regressão logística é equivalente a uma rede neural de uma única camada sem camadas ocultas:
- Combinação linear: z = W·x + b
- Ativação sigmoid: ŷ = σ(z) = 1 / (1 + e⁻ᶻ), com saída em [0, 1]
- A fronteira de decisão é um hiperplano linear no espaço de features

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

## Avaliação do modelo (100 features)

In [ ]:
y_pred = model.predict(X_test)

ac = accuracy_score(y_test, y_pred)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

sensitivity = tp / (tp + fn) # recall da classe câncer
specificity = tn / (tn + fp) # recall da classe normal

print(f"Acurácia: {ac:.4f} ({ac*100:.2f}%)")
print(f"Sensibilidade: {sensitivity:.4f} ({sensitivity*100:.2f}%)")
print(f"Especificidade: {specificity:.4f} ({specificity*100:.2f}%)")

cm = confusion_matrix(y_test, y_pred)
print(f"\nMatriz de Confusão:\n{cm}")

### Visualização da Matriz de Confusão (100 features)

In [ ]:
labels = ['Câncer', 'Normal']

plt.figure(figsize=(7, 6))
sns.heatmap(cm, cmap='YlOrRd', annot=True, fmt='d', cbar=False,
            xticklabels=labels, yticklabels=labels)
plt.title('Matriz de Confusão - Regressão Logística (100 features)', fontsize=13)
plt.xlabel('Predito')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

## Seleção de características com ANOVA F-score

Vamos usar ANOVA F-score para selecionar as 50 features com maior poder discriminativo.
Ela mede o quanto cada feature separa as classes, comparando a variância entre grupos com a variância dentro de cada grupo.

In [ ]:
selector = SelectKBest(score_func=f_classif, k=50)
X_train_50 = selector.fit_transform(X_train, y_train)
X_test_50  = selector.transform(X_test)

print(f"Índices das features selecionadas: {np.where(selector.get_support())[0]}")

In [ ]:
# modelo apenas com as 50 características mais relevantes
model_50 = LogisticRegression(max_iter=1000, random_state=42)
model_50.fit(X_train_50, y_train)
y_pred_50 = model_50.predict(X_test_50)

In [ ]:
ac_50 = accuracy_score(y_test, y_pred_50)
tn2, fp2, fn2, tp2 = confusion_matrix(y_test, y_pred_50).ravel()
sensitivity_50 = tp2 / (tp2 + fn2)
specificity_50 = tn2 / (tn2 + fp2)
cm_50 = confusion_matrix(y_test, y_pred_50)

print(f"Acurácia: {ac_50:.4f} ({ac_50*100:.2f}%)")
print(f"Sensibilidade: {sensitivity_50:.4f} ({sensitivity_50*100:.2f}%)")
print(f"Especificidade: {specificity_50:.4f} ({specificity_50*100:.2f}%)")
print(f"\nMatriz de Confusão:\n{cm_50}")

### Visualização da Matriz de Confusão (50 features)

In [ ]:
plt.figure(figsize=(7, 6))
sns.heatmap(cm_50, cmap='YlOrRd', annot=True, fmt='d', cbar=False,
            xticklabels=labels, yticklabels=labels)
plt.title('Matriz de Confusão - Regressão Logística (50 features)', fontsize=13)
plt.xlabel('Predito')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

## Comparação entre os dois modelos

| Métrica | 100 features | 50 features |
|---|---|---|
| **Acurácia** | **95,45%** | 88,64% |
| **Sensibilidade** | **92,00%** | 88,00% |
| **Especificidade** | **100,00%** | 89,47% |



Com base nos resultados das métricas, concluímos que o modelo com todas as 100 features apresentou desempenho superior.  
A especificidade de 100% indica que nenhum paciente normal foi classificado erroneamente como canceroso.

Porém, no modelo com apenas 50 features, houve uma leve queda de desempenho, sugerindo que as features descartadas também carregavam informações relevantes para uma classificação bem sucedida.